# 明日の柏、いちばん暖かいのは何時？

**氏名：［自分の氏名に書き換える］**

Week 03 Homework｜データで見つけた、小さな発見

以下は完成例です。自分の問い・データ・観察に合わせて変更してください。提出するNotebookの一番上には、作品タイトルと氏名を必ず書きます。

## 私の問い

明日の柏は、何時にいちばん暖かくなるだろう？ 朝と昼の気温差も知りたい。

- **データ：** Open-Meteoの1時間ごとの気温予報
- **出典：** [Open-Meteo Weather Forecast API](https://open-meteo.com/en/docs)
- **場所：** 柏付近（緯度35.86、経度139.97）
- **単位・時刻：** ℃・日本時間（Asia/Tokyo）

予報データなので、実測値ではありません。

## 1｜APIからJSONを読む

読み込み部分はそのまま使ってOK。URLの`latitude`と`longitude`が場所、`forecast_days=2`が今日と明日の2日分です。

通信できない場合は、同じフォルダに置いた`weather-kashiwa-example.json`を読みます。これは2026年10月15日に取得した保存例で、最新の予報ではありません。

In [1]:
import json
from urllib.request import urlopen
from urllib.error import URLError
from datetime import datetime, timezone, timedelta

url = "https://api.open-meteo.com/v1/forecast?latitude=35.86&longitude=139.97&hourly=temperature_2m&forecast_days=2&timezone=Asia%2FTokyo"

try:
    with urlopen(url, timeout=20) as response:
        data = json.load(response)
    retrieved_at = datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="seconds")
    source_label = "APIから取得した予報"
except (URLError, TimeoutError):
    with open("weather-kashiwa-example.json", encoding="utf-8") as file:
        data = json.load(file)
    retrieved_at = "2026-10-15（保存データの取得日）"
    source_label = "保存例：最新の予報ではありません"

print(source_label)
print(f"取得日時: {retrieved_at}")
print(f"出典URL: {url}")

APIから取得した予報
取得日時: 2026-10-15T08:26:56+09:00
出典URL: https://api.open-meteo.com/v1/forecast?latitude=35.86&longitude=139.97&hourly=temperature_2m&forecast_days=2&timezone=Asia%2FTokyo


## 2｜辞書の中の、二つのリスト

`data`は辞書。`data["hourly"]`も辞書。その中の`time`と`temperature_2m`はリストです。

**同じ添字の時刻と気温が、一組のデータ。**

In [2]:
times = data["hourly"]["time"]
temperatures = data["hourly"]["temperature_2m"]
unit = data["hourly_units"]["temperature_2m"]

print(type(data))
print(type(data["hourly"]))
print(type(temperatures))
print(f"時刻: {times[0]} / 気温: {temperatures[0]} {unit}")
print(f"時刻は{len(times)}件、気温は{len(temperatures)}件")

<class 'dict'>
<class 'dict'>
<class 'list'>
時刻: 2026-10-15T00:00 / 気温: 18.8 °C
時刻は48件、気温は48件


## 3｜明日の24時間だけ取り出す

このURLでは、今日の0時から48時間分が届きます。`[24:48]`で明日の0〜23時を取り出します。

先頭の添字は24。終点の48は含みません。

In [3]:
tomorrow_times = times[24:48]
tomorrow_temperatures = temperatures[24:48]

print(f"対象日: {tomorrow_times[0][:10]}")
print(f"最初: {tomorrow_times[0]} / {tomorrow_temperatures[0]} {unit}")
print(f"最後: {tomorrow_times[-1]} / {tomorrow_temperatures[-1]} {unit}")
print(f"比較する時間数: {len(tomorrow_temperatures)}")

対象日: 2026-10-16
最初: 2026-10-16T00:00 / 17.0 °C
最後: 2026-10-16T23:00 / 18.5 °C
比較する時間数: 24


## 4｜問いに答える

`max()`で最高気温、`min()`で最低気温を求めます。`.index()`は、その値が最初に見つかる添字を返します。

その添字で時刻のリストを読むと、「何時」が分かります。同じ最高気温が複数回あるときは、最初の時刻を選びます。

In [4]:
warmest = max(tomorrow_temperatures)
coldest = min(tomorrow_temperatures)
warmest_index = tomorrow_temperatures.index(warmest)
coldest_index = tomorrow_temperatures.index(coldest)
difference = round(warmest - coldest, 1)

print("明日の柏の気温予報")
print(f"最も暖かい: {tomorrow_times[warmest_index]} / {warmest} {unit}")
print(f"最も寒い: {tomorrow_times[coldest_index]} / {coldest} {unit}")
print(f"一日の気温差: {difference} {unit}")

明日の柏の気温予報
最も暖かい: 2026-10-16T14:00 / 26.6 °C
最も寒い: 2026-10-16T06:00 / 15.4 °C
一日の気温差: 11.2 °C


## 5｜元データで確かめる

明日だけのリストではなく、元の48件のリストに戻って確認します。明日の先頭は添字24なので、24を足します。

この時刻と気温が、上の結果と一致するか見てください。

In [5]:
original_index = 24 + warmest_index

print(f"元データの添字: {original_index}")
print(f"元データの時刻: {data['hourly']['time'][original_index]}")
print(f"元データの気温: {data['hourly']['temperature_2m'][original_index]} {unit}")
print(f"最高気温の一致: {data['hourly']['temperature_2m'][original_index] == warmest}")

元データの添字: 38
元データの時刻: 2026-10-16T14:00
元データの気温: 26.6 °C
最高気温の一致: True


## 小さな発見

**以下は2026年10月15日の保存データを使った観察例です。再実行したら、自分の出力に合わせて書き直してください。**

**分かったこと：** 10月16日の柏は、14時の予報が26.6℃で最も暖かい。最低は6時の15.4℃で、一日の差は11.2℃あった。「明日の気温」も、一つの数字だけでは朝と昼の違いが見えない。

**確かめた値：** 元のJSONの添字38を読み、時刻が`2026-10-16T14:00`、気温が`26.6`であることを確認した。

**限界：** これは柏付近の予報で、自分の家やキャンパスでの実測ではない。風・雨・体感温度は調べていない。予報は更新されるので、取得日が違えば結果も変わる。

**AI利用の記録：** この例を参考にしたことと、ほかにAIを使った場合は用途を書く。

## 自分の課題にする

気になるAPIと問いに変え、結果と自分の観察を見せてください。文章で結果を表示するだけでもOK。グラフや地図は任意です。

**提出前：** タイトル・氏名を記入 → 全Cellを上から実行 → 出力と観察を保存。

**締切：2026年10月21日（水）23:59 JST。** `.ipynb`と、読み込みに使ったJSONなどのファイルをUNIPAへ。スターターの使用は任意です。